# Multi-Agent Workflows

Researcher → Analyst → Writer pipeline built with the OpenAI Agents SDK.

The tool, models and agents live in `research_agents.py`, which this notebook shares with the Streamlit app (`multi_agent_app.py`). Edit agents there.

## 1. Environment setup

In [ ]:
import os

from research_agents import (
    MODEL, analyst, researcher, writer, tavily_search,
    run_agent, run_tool, run_pipeline, run_manager,
)

for key in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    value = os.getenv(key)
    status = "set" if value and "here" not in value else "MISSING / placeholder"
    print(f"{key}: {status}")

print("\nModel:", MODEL)
for agent in (researcher, analyst, writer):
    print(f"{agent.name:<10} tools={[t.name for t in agent.tools]}  output={agent.output_type.__name__}")

## 2. Tavily web search tool

Defined in `research_agents.py`: POST to `https://api.tavily.com/search`, `query` + `max_results` (TypedDict, default 2).

In [ ]:
import json

print(json.dumps(tavily_search.params_json_schema, indent=2))
print(run_tool(tavily_search, "Latest news on quantum computing", max_results=2))

## 3. Researcher and Analyst

In [ ]:
QUERY = "Why is Labubu so popular and what are the rarest Labubu collectibles?"

research = run_agent(researcher, QUERY).final_output  # ResearchOutput
print(research.summary)

In [ ]:
analysis = run_agent(analyst, research.summary).final_output  # ResearchOutput
print(analysis.summary)

## 4. Full pipeline: Researcher → Analyst → Writer

`run_manager()` runs the async `manager_run()` with all three agents sharing one `SQLiteSession`.

In [ ]:
from IPython.display import Markdown, display

TEST_QUERY = "What is the public sentiment and expert reviews about the Tesla Cybertruck?"
final_report = run_manager(TEST_QUERY)

display(Markdown("## Short summary\n\n" + final_report.short_summary))
display(Markdown("## Full report\n\n" + final_report.markdown_report))
questions = "\n".join(f"{i}. {q}" for i, q in enumerate(final_report.follow_up_questions, 1))
display(Markdown("## Follow-up questions\n\n" + questions))
print(f"\nReport length: {len(final_report.markdown_report.split())} words")